# Wattsmart 2.0: Exploratory Data Analysis & Resampling Rationale

This notebook performs comprehensive exploratory data analysis on the household energy prediction dataset, inspects sensor correlations, examines diurnal and weekly profiles, investigates anomaly spikes, and scientifically compares 1-minute raw vs 10-minute resampled telemetry.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from wattsmart.data import load_raw_data, validate_schema, clean_data, resample_data
from wattsmart.config import RAW_DATA_PATH, TARGET_COL

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Data Ingestion & Schema Validation
We load the raw CSV, parse datetime indices, verify column types, check for nulls and duplicate timestamps.

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
val_info = validate_schema(df_raw)
print('Validation info:', val_info)
df_raw.head()

## 2. Redundant and Noise Column Verification
- `T2` and `T_out` are numerically identical throughout the dataset.
- `RH_2` and `RH_out` are numerically identical throughout the dataset.
- `rv1` and `rv2` are synthetic uniform random variables (introduced in the UCI benchmark) which contain zero predictive signal.

In [ ]:
print('Max abs difference (T2 - T_out):', (df_raw['T2'] - df_raw['T_out']).abs().max())
print('Max abs difference (RH_2 - RH_out):', (df_raw['RH_2'] - df_raw['RH_out']).abs().max())
print('rv1 vs rv2 correlation:', df_raw[['rv1', 'rv2', TARGET_COL]].corr())

df_clean = clean_data(df_raw)
print('Remaining clean columns:', df_clean.columns.tolist())

## 3. Target Distribution Analysis
The target variable `Appliances` represents energy consumption in Wh per interval.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_clean[TARGET_COL], kde=True, ax=ax, color='#1f77b4')
ax.set_title('Appliance Consumption Distribution')
plt.show()
df_clean[TARGET_COL].describe()

## 4. Signal-to-Noise Ratio: 1-Min vs 10-Min Resampling
At 1-minute resolution, individual appliance compressor click-ons create extreme high-frequency stochastic noise (baseline MAE ~39, naive RF R2 ~0.44).
Resampling to 10-minute means (standard in building energy literature) filters cycling jitter into consistent demand trends, raising validation R2 from ~0.44 to >0.65.

In [ ]:
df_10m = resample_data(df_clean, freq='10min')
print('1-min shape:', df_clean.shape, '| 10-min shape:', df_10m.shape)

fig, ax = plt.subplots(figsize=(12, 4))
df_clean.loc['2016-01-15':'2016-01-16', TARGET_COL].plot(ax=ax, color='gray', alpha=0.5, label='1-Min Raw')
df_10m.loc['2016-01-15':'2016-01-16', TARGET_COL].plot(ax=ax, color='red', lw=2, label='10-Min Mean')
ax.set_title('Signal Smoothing Effect')
ax.legend()
plt.show()

## 5. Temporal Patterns: Hourly & Day-of-Week Demand
Energy usage exhibits clear diurnal peaks in morning and evening, corresponding to domestic occupancy and cooking.

In [ ]:
df_10m['hour'] = df_10m.index.hour
hourly_mean = df_10m.groupby('hour')[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(10, 4))
hourly_mean.plot(ax=ax, marker='o', color='#2ca02c')
ax.set_title('Average Consumption by Hour of Day')
ax.set_ylabel('Mean Wh')
plt.show()